In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Scenario

TheLook, a hypothetical eCommerce clothing retailer, stores data on customers, products, orders, logistics, web events, and digital marketing campaigns in BigQuery. The company wants to leverage the team's existing SQL and PySpark expertise to analyze this data using Apache Spark.

To avoid manual infrastructure provisioning or tuning for Spark, TheLook seeks an auto-scaling solution that allows them to focus on workloads rather than cluster management. Additionally, they want to minimize the effort required to integrate Spark and BigQuery while staying within the BigQuery Studio environment, possibly using BigQuery notebooks.

Then, they want to productionalize the model by having a queryable endpoint, and a way to query it via natural language.

In this notebook, you implement a workflow to predict whether a user will make a purchase by building a Logistic Regression Classifier using PySpark and leveraging BigQuery Studio's native notebook integration and AI-features for exploring the data. Then, you deploy this model to an inference server and create an agent to query the model using natural language.

Original source for this notebook: [GitHub](https://github.com/GoogleCloudPlatform/devrel-demos/blob/main/data-analytics/qwiklabs/Spark_Data_Science.ipynb)

# **Task 1. Access the notebook in BigQuery**

You have already completed this task in BigQuery to open the notebook and connect to a runtime.

To run each cell for the remaining tasks, click on run (black play button icon) on the top right of the cell.

# **Task 2. Configure your notebook environment**

In this task, you configure your notebook environment with the necessary tools and items to complete the lab including configuring [Private Google Access](https://docs.cloud.google.com/vpc/docs/configure-private-google-access#gcloud_1), creating a Cloud Storage bucket, and creating a BigQuery dataset.

Add the project id and region as provided on the lab instruction page.


In [ ]:
PROJECT_ID = "" # @param {type:"string"}

REGION = "" # @param {type:"string"}

To use [Google Cloud Serverless for Apache Spark](https://docs.cloud.google.com/dataproc-serverless/docs), turn on [Private Google Access](https://docs.cloud.google.com/vpc/docs/configure-private-google-access#gcloud_1).

In [ ]:
import subprocess

command = [
    "gcloud",
    "compute",
    "networks",
    "subnets",
    "update",
    "default",
    f"--region={REGION}",
    "--enable-private-ip-google-access"
]

result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Create a Cloud Storage bucket.


In [ ]:
from google.cloud import storage
from google.cloud.exceptions import NotFound

BUCKET_NAME = f"{PROJECT_ID}-demo"

storage_client = storage.Client(project=PROJECT_ID)
try:
    bucket = storage_client.get_bucket(BUCKET_NAME)
    print(f"Bucket {BUCKET_NAME} already exists.")
except NotFound:
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Bucket {BUCKET_NAME} created.")

Create a BigQuery dataset.

In [ ]:
from google.cloud import bigquery

DATASET_ID = f"{PROJECT_ID}.demo"

client = bigquery.Client()

dataset = bigquery.Dataset(DATASET_ID)

dataset.location = REGION

dataset = client.create_dataset(dataset, exists_ok=True)

# **Task 3: Create a connection to Google Cloud Serverless for Apache Spark**

In this task, you use Spark Connect to a serverless Spark session to run interactive Spark jobs by completing the following steps:

* Set up the Spark environment: It imports necessary libraries for connecting to Dataproc and using PySpark.
* Configure the Dataproc session: It creates and configures a Spark Session with the necessary parameters, providing the spark object for subsequent Spark operations.



Create the `Session()` object to definite parameters. Here, you set the runtime version to `3.0` and cap the number of executors (Spark workers)to `4`.

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

session = Session()

session.runtime_config.version = "3.0"

# To avoid going over quota in this demo, cap the max number of Spark workers.
session.runtime_config.properties = {
    "spark.dynamicAllocation.maxExecutors": "4"
}

For large workloads, you can use [Lightning Engine](https://cloud.google.com/products/lightning-engine) to further accelerate your Spark workloads by up to [4.3x](https://docs.cloud.google.com/dataproc-serverless/docs/guides/lightning-engine#enable_lightning_engine). The dataset in this lab is too small to see meaningful differences. For reference, you could use the following code with your own notebook and data:

```
session.runtime_config.properties = {
  "dataproc.runtime": "premium",
  "spark.dataproc.engine": "lightningEngine",
}


Create the Spark Session.

**Notes:**
1. The first time you run this cell, you may encounter a service account error. Wait 1 minute and try again.

2. You may be asked to authenticate. Click the link in the output to do so, and then wait 2 minutes for permissions to propagate before running the cell again. You may need to **run the cell multiple times** for it to execute successfully, as more time may be needed for the permissions to propagate.

When you see the output message that `Dataproc Session was successfully created`, you can proceed to the next task.

In [ ]:
spark = (
    DataprocSparkSession.builder
      .appName("CustomSparkSession")
      .dataprocSessionConfig(session)
      .getOrCreate()
)

# **Task 4. Load data and explore with Gemini**

In this task, you run through the first important step in any data science project: preparing your data. You start by loading data into an Apache Spark dataframe from BigQuery. Then, you use Gemini to generate PySpark code to explore the data.

The dataset is a synthetic dataset of customer information and purchase history, which is available as a BigQuery Public Dataset called [thelook_ecommerce]((https://console.cloud.google.com/marketplace/product/bigquery-public-data/thelook-ecommerce).

The serverless Spark runtime is configured to load your data directly from BigQuery into a Spark dataframe.

In [ ]:
users = spark.read.format("bigquery").option("table", "bigquery-public-data.thelook_ecommerce.users").load()
users.show()

Load the order_items data.

In [ ]:
order_items = spark.read.format("bigquery").option("table", "bigquery-public-data.thelook_ecommerce.order_items").load()
order_items.show()

Register these tables as Spark SQL tables.

In [ ]:
users.createOrReplaceTempView("users")
order_items.createOrReplaceTempView("order_items")

Use SparkSQL to query a table directly. Query the `users` table.

In [ ]:
spark.sql("SELECT * FROM users LIMIT 10").show()

Query the `order_items` table.

In [ ]:
spark.sql("SELECT * FROM order_items LIMIT 10").show()

BigQuery Studio can leverage Gemini for [advanced code completion capabilities](https://cloud.google.com/bigquery/docs/write-sql-gemini#generate_python_code?utm_campaign=CDR_0x225cfd13_default_b407565440&utm_source=external&utm_medium=web) which can use Natual Language to perform exploratory analysis using SQL and even generate PySpark Code for Feature Engineering.

## Complete the following code cells to generate query code with Gemini

In each code cell below a prompt, click **generate with AI**. Paste each prompt text into the open text box, and click **Generate**. After the code has been generated and populated into the code cell, run the cell.

**Note:** If you receive an error when running the cell, repeat the step to generate the code with Gemini to produce new code, and run the cell again. You may need to **generate multiple times** for the generated code to run successfully.

**Prompt 1**: Using PySpark, explore the users table and show the first 10 rows.

**Prompt 2**: Using PySpark, explore the order_items table and show the first 10 rows.

**Prompt 3**: Using PySpark, show the top 5 most frequent countries in the users table. Display the country and the number of users from each country.

**Prompt 4**: Using PySpark, find the average sale price of items in the order_items table.

**Prompt 5**: Using the table "users", generate code to plot country vs traffic source using a suitable plotting library.

**Prompt 6:** Create a histogram showing the distribution of "age", "country", "gender", "traffic_source".

## **Task 5. Perform feature engineering**

In this task, you perform feature engineering on the data. Specifically, you select the appropriate columns, transform the data into more suitable data types, and identify a label column.

In this steps below, you derive two key columns from the input data:
* **Creation of features column**:
It combines user attributes (age, hashed categorical features) into a numerical array, preparing them for a machine learning model.
* **Generation of label column:**
It creates a binary target variable indicating whether a user has made a purchase or not, derived from order information.

In [ ]:
features = spark.sql("""
SELECT
  CAST(u.age AS DOUBLE) AS age,
  CAST(hash(u.country) AS BIGINT) * 1.0 AS country_hash,
  CAST(hash(u.gender) AS BIGINT) * 1.0 AS gender_hash,
  CAST(hash(u.traffic_source) AS BIGINT) * 1.0 AS traffic_source_hash,
  CASE WHEN COUNT(oi.id) > 0 THEN 1 ELSE 0 END AS label -- Changed label generation to count order items
FROM users AS u
LEFT JOIN order_items AS oi
ON u.id = oi.user_id
GROUP BY u.id, u.age, u.country, u.gender, u.traffic_source
""")
features.show()

# **Task 6. Train a logistic regression model**

In this task, you use MLlib to train a logistic regression model. First, you use a `VectorAssembler` to convert the data into a vector format, and then `StandardScaler` scales the features column for better performance. Next, you create a reference to a `LogisticRegression` model and define hyperparamters. Last, you combine these steps into a `Pipeline` object, train the model using the `fit()` function and transform the data using the `transform()` function.

This code trains a logistic regression model to predict user purchase behavior with these steps:
* **Vector Assembly:** VectorAssembler formats the data into readable vectors.
* **Feature Scaling:** StandardScaler scales the "features" column.
* **Model Initialization:** LogisticRegression is set up to predict the binary "label" (purchase/no purchase), with hyperparameters for training.
* **Pipeline Definition:** A Pipeline chains StandardScaler and LogisticRegression for streamlined scaling and training.
* **Model Training:** `pipeline.fit(dataset)` trains the pipeline (scaling and then the model).
* **Prediction:** `pipeline_model.transform(dataset)` generates predictions, and `transformed_dataset.show()` displays the results.

In short, these steps scale features, train a logistic regression model within a pipeline, and produce purchase predictions.

In [ ]:
from pyspark.ml.classification import LogisticRegression, LogisticRegressionModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.functions import array_to_vector

#Split Train and Test Data (80:20)
train_data, test_data = features.randomSplit([0.8, 0.2], seed=42)

In [ ]:
# Initialize VectorAssembler
assembler = VectorAssembler(
    inputCols=["age", "country_hash", "gender_hash", "traffic_source_hash"],
    outputCol="assembled_features"
)

# Initialize StandardScaler
scaler = StandardScaler(inputCol="assembled_features", outputCol="scaled_features")

# Initialize Logistic Regression model
lr = LogisticRegression(
    maxIter=100,
    regParam=0.2,
    threshold=0.8,
    featuresCol="scaled_features",
    labelCol="label"
)

# Define pipeline
pipeline = Pipeline(stages=[assembler, scaler, lr])

# Fit the model
pipeline_model = pipeline.fit(train_data)

# Transform the dataset using the trained model
transformed_dataset = pipeline_model.transform(test_data)

In [ ]:
transformed_dataset.show()

# **Task 7. Conduct model evaluation**

In this task, you evaluate your newly transformed dataset and generate the evaluation metric [area under curve (AUC)](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc#area_under_the_curve_auc). Then, you use Gemini to generate PySpark code to visualize your model output.

This code evaluates the trained model's performance by:
* **Initializing an Evaluator:** A BinaryClassificationEvaluator is set up to calculate the Area Under the Precision-Recall Curve (AUC-PR).
* **Calculating AUC-PR:** The evaluate() method calculates the AUC-PR score using the model's predictions.

These steps quantify the model's ability to distinguish between the two classes (e.g., purchase/no purchase).

In [ ]:
# Model evaluation
eva = BinaryClassificationEvaluator(metricName="areaUnderPR")
aucPR = eva.evaluate(transformed_dataset)
print(f"AUC PR: {aucPR}")

## Complete the following code cells to generate code to visualize the output with Gemini

In each code cell below a prompt, click **generate with AI**. Paste each prompt text into the open text box, and click **Generate**. After the code has been generated and populated into the code cell, run the cell.

**Note:** If you receive an error when running the cell, repeat the step to generate the code with Gemini to produce new code, and run the cell again. You may need to **generate multiple times** for the generated code to run successfully.

**Prompt 1:** Generate code to plot the Precision-Recall (PR) curve. Calculate precision and recall from the model's predictions and display the PR curve using a suitable plotting library.

**Prompt 2:** Generate code to create a confusion matrix visualization. Calculate the confusion matrix from the model's predictions and display it as a heatmap or a table with counts of true positives (TP), true negatives (TN), false positives (FP), and false negatives (FN).

# **Task 8. Write predictions to BigQuery**

In this task, you use Gemini to generate code to write your predictions to a new table in your BigQuery dataset.

## Complete the following code cell to generate code to write predictions to BigQuery with Gemini.

In each code cell below a prompt, click **generate with AI**. Paste each prompt text into the open text box, and click **Generate**. After the code has been generated and populated into the code cell, run the cell.

**Note:** If you receive an error when running the cell, repeat the step to generate the code with Gemini to produce new code, and run the cell again. You may need to **generate multiple times** for the generated code to run successfully.

**Prompt:** Using Spark, write the transformed dataset to BigQuery.

# **Task 9. Save the model to Cloud Storage**

In this task, you use MLlib's native functionality to save your model to Cloud Storage, so that the inference server can load the model from a bucket.

In [ ]:
GCS_MODEL_PATH="models/prediction_model"
pipeline_model.write().overwrite().save(f"gs://{BUCKET_NAME}/{GCS_MODEL_PATH}")

# **Task 10. Create an inference server**

Cloud Run is a flexible tool to run serverless web apps. It uses Docker containers to provide users with maximum customability.

In this task, you use [Cloud Run](https://cloud.google.com/run) to create an inference server for your MLlib model. Specifically, you use a [Dockerfile](https://github.com/GoogleCloudPlatform/devrel-demos/blob/main/data-analytics/dataproc-webinar/data-science/inference-server/Dockerfile) that is configured to run a Flask app powering PySpark. This container runs on Cloud Run to perform inference on input data.

For reference, the original source code can be found on [GitHub](https://github.com/GoogleCloudPlatform/devrel-demos/tree/main/data-analytics/dataproc-webinar/data-science/inference-server).

Copy the files from a Cloud Storage bucket to the notebook environment.

In [ ]:
!gcloud storage cp -r gs://spls/gsp1374/ /content

Deploy the Cloud Run server. This step can take up to ten minutes - make sure to keep the notebook open until this cell finishes running.

In [ ]:
import subprocess

command = [
    "gcloud",
    "run",
    "deploy",
    "inference-server",
    "--source",
    "/content/gsp1374/data-analytics/dataproc-webinar/data-science/inference-server",
    "--region",
    f"{REGION}",
    "--port",
    "8080",
    "--memory",
    "2Gi",
    "--allow-unauthenticated",
    "--set-env-vars",
    f"GCS_BUCKET={BUCKET_NAME},GCS_MODEL_PATH={GCS_MODEL_PATH}",
    "--startup-probe",
    "tcpSocket.port=8080,initialDelaySeconds=240,failureThreshold=3,timeoutSeconds=240,periodSeconds=240"
]

result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Copy the Service URL from the output and paste it below.

In [ ]:
INFERENCE_SERVER_URL = "" #@param {type:"string"}

Test your inference server.

In [ ]:
import requests

age = "25.0"
country = "United States"
traffic_source = "Search"
gender = "F"

response = requests.post(
    f"{INFERENCE_SERVER_URL}/predict",
    json=[{"age": age, "country": country, "traffic_source": traffic_source, "gender": gender}],
    headers={"Content-Type": "application/json"}
)

print(response.json())

# **Task 11. Configure an agent**

[Vertex AI Agent Engine](https://docs.cloud.google.com/agent-builder/agent-engine/overview) is a part of Vertex AI Platform, a set of services that enable developers to deploy, manage, and scale AI agents in production. It has many tools including evaluating agents, session contexts, and code execution.

It supports many popular agentic frameworks, including the [Agent Development Kit (ADK)](https://google.github.io/adk-docs/). ADK is an open source agentic framework that, while built and optimized for use with Gemini and the Google ecosystem, is model-agnotic. It is designed to make agent development feel more like software development.

In this task, you use both of these tools to build an agent that can call the deployed SparkML model.


Initialize a Vertex AI Client.

In [ ]:
import vertexai
from vertexai import agent_engines # For the prebuilt templates

client = vertexai.Client(  # For service interactions via client.agent_engines
    project=f"{PROJECT_ID}",
    location=f"{REGION}",
)

Define a function for querying the deployed model.

In [ ]:
def predict_purchase(
    age: str = "25.0",
    country: str = "United States",
    traffic_source: str = "Search",
    gender: str = "M",
):
    """Predicts whether or not a user will purchase a product.

    Args:
        age: The age of the user.
        country: The country of the user. One of: "China", "Poland", "Germany", "United States", "Spain", "United Kingdom", "España", "Japan", "Brasil", "Colombia", "Belgium", "South Korea", "Austria", "France", "Australia".
        Traffic_source: The source of the user's traffic. One of: "Display", "Email", "Search", "Organic", "Facebook".
        gender: The gender of the user. One of: "M" or "F".

    Returns:
        True if the model output is 1.0, False otherwise.
    """
    import requests
    response = requests.post(
        f"{INFERENCE_SERVER_URL}/predict",
        json=[{"age": age, "country": country, "traffic_source": traffic_source, "gender": gender}],
        headers={"Content-Type": "application/json"}
    )
    return response.json()

Test the function by passing in sample parameters.

In [ ]:
predict_purchase(age=25.0, country="United States", traffic_source="Search", gender="M")

Using the ADK, define an agent below and provide the `predict_purchase` function as a tool.

In [ ]:
from google.adk.agents import Agent
from vertexai import agent_engines

agent = Agent(
   model="GEMINI_FLASH_MODEL_ID",
   name='purchase_prediction_agent',
   tools=[predict_purchase]
)

Test the agent locally by passing in a query.

In [ ]:
app = agent_engines.AdkApp(agent=agent)
async for event in app.async_stream_query(
    user_id="123",
    message="Will a 25 yo male from the United States who came from Search make a purchase? Strictly output 'yes' or 'no'.",
):
    try:
        print(event['content']['parts'][0]['text'])
    except:
      continue

Deploy the model to Agent Engine.

In [ ]:
remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "requirements": ["google-cloud-aiplatform[agent_engines,adk]"],
        "staging_bucket": f"gs://{BUCKET_NAME}",
        "display_name": "purchase-predictor",
        "description": "Agent that predicts whether or not a user will purchase a product.",
    }
)

[Optional] Once done, you can view the deployed model in the [Cloud Console](https://console.cloud.google.com/vertex-ai/agents/agent-engines).

Last, query the model again. The code now uses the deployed agent, instead of the local version.

In [ ]:
async for event in remote_agent.async_stream_query(
    user_id="123",
    message="Will a 25 yo male from the United States who came from Search make a purchase? Strictly output 'yes' or 'no'.",
):
    try:
        print(event['content']['parts'][0]['text'])
    except:
      continue

Congratulations! By executing the steps in this notebook, you learned how to use BigQuery Studio notebooks to run data science workflows that leverage Google Cloud Serverless for Apache Spark to train and evaluate a classification model. You also deployed an inference server for the model and deployed an agent to query the inference server using natural language with Agent Engine and the Agent Development Kit (ADK).